
# Health & Wellbeing Trajectory Pipeline

This notebook builds a **resident-month classification pipeline** that predicts whether a resident is likely to show a **positive health trajectory in the next month**.

The notebook is organized to match the full IS 455 pipeline story:
1. Problem Framing  
2. Data Acquisition, Preparation & Exploration  
3. Modeling & Feature Selection  
4. Evaluation & Interpretation  
5. Causal and Relationship Analysis  
6. Deployment Notes

This version is also written to be more robust than the earlier draft:
- clearer markdown sections throughout
- stronger imports and setup in the first code cell
- safer handling of health-score scale differences
- safer handling of class-balance problems during cross-validation
- outputs saved under `generated_outputs`



## 1. Problem Framing

### Business question
Which resident-month records appear most likely to lead to a **positive health and wellbeing trajectory next month**?

### Why this matters
Health and wellbeing are central resident outcomes. Staff need a way to identify who may need more support before a negative trajectory becomes more visible.

### Who cares about this?
This pipeline is useful for:
- case-management staff
- administrators monitoring resident wellbeing
- leadership reviewing aggregate health outcomes across safehouses

### Predictive vs. explanatory choice
This notebook is **primarily predictive**. The goal is to forecast next-month wellbeing outcomes using current-month information.

The notebook still includes a relationship-analysis section, but those results should be interpreted as **associational**, not causal.

### Suggested website use
This pipeline fits naturally into:
- **Caseload Inventory**
- **Admin Dashboard**
- **Reports & Analytics**



## 2. Data Acquisition, Preparation & Exploration

This section:
- loads the relevant INTEX tables
- standardizes likely schema variants
- builds a resident-month health panel
- adds supporting case-management signals from incidents, counseling, and intervention plans

The preparation approach follows the project pattern:
- robust file loading
- standard naming
- repeatable monthly aggregation
- target creation that avoids future leakage


In [3]:
# Imports, configuration, and helper functions

from pathlib import Path
import warnings
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit, StratifiedKFold, cross_validate
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report,
)
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")

# Save notebook outputs under generated_outputs inside ml-pipelines.
OUTPUT_DIR = Path("./generated_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Resolve the most likely raw-data folder for the INTEX project.
DATA_DIR_CANDIDATES = [
    Path("./lighthouse_csv_v7/lighthouse_csv_v7"),
    Path("./lighthouse_csv_v7"),
    Path("../lighthouse_csv_v7/lighthouse_csv_v7"),
    Path("../lighthouse_csv_v7"),
]

def resolve_data_dir(candidates):
    # Prefer folders that directly contain CSV files.
    for candidate in candidates:
        if candidate.exists() and any(candidate.glob("*.csv")):
            return candidate

    # If a parent folder exists, drill down one level into a nested CSV folder.
    for candidate in candidates:
        if candidate.exists():
            for sub in candidate.iterdir():
                if sub.is_dir() and any(sub.glob("*.csv")):
                    return sub

    raise FileNotFoundError(
        "Could not find a data folder containing CSV files. Checked: "
        + ", ".join(str(p) for p in candidates)
    )

DATA_DIR = resolve_data_dir(DATA_DIR_CANDIDATES)

print("Output dir:", OUTPUT_DIR.resolve())
print("Resolved DATA_DIR:", DATA_DIR.resolve())

# Define the source files needed for this pipeline.
TABLE_FILE_MAP = {
    "residents": "residents.csv",
    "safehouses": "safehouses.csv",
    "health_wellbeing_records": "health_wellbeing_records.csv",
    "incident_reports": "incident_reports.csv",
    "process_recordings": "process_recordings.csv",
    "intervention_plans": "intervention_plans.csv",
}

def month_floor(series: pd.Series) -> pd.Series:
    # Normalize date-like values to the first day of the month.
    return pd.to_datetime(series, errors="coerce").dt.to_period("M").dt.to_timestamp()

def load_tables(data_dir: Path, file_map: dict[str, str]) -> dict[str, pd.DataFrame]:
    # Load each table if it exists, otherwise return an empty DataFrame for visibility.
    tables = {}
    for table_name, file_name in file_map.items():
        path = data_dir / file_name
        if path.exists():
            tables[table_name] = pd.read_csv(path)
            print(f"Loaded {table_name}: {tables[table_name].shape} from {path}")
        else:
            tables[table_name] = pd.DataFrame()
            print(f"Missing {table_name}: expected {path}")
    return tables

def rename_if_present(df: pd.DataFrame, rename_map: dict[str, str]) -> pd.DataFrame:
    # Rename only the columns that actually exist in the current file version.
    available = {old: new for old, new in rename_map.items() if old in df.columns}
    if available:
        df = df.rename(columns=available)
    return df

def safe_numeric(series: pd.Series) -> pd.Series:
    # Convert values to numeric while coercing bad values to NaN.
    return pd.to_numeric(series, errors="coerce")

tables = load_tables(DATA_DIR, TABLE_FILE_MAP)

if tables["health_wellbeing_records"].empty:
    raise FileNotFoundError(
        "health_wellbeing_records.csv was not found. Check DATA_DIR and the printed file list above."
    )

Output dir: C:\Users\Ashns\OneDrive\INTEX26\INTEX_W2026\ml-pipelines\generated_outputs
Resolved DATA_DIR: C:\Users\Ashns\OneDrive\INTEX26\INTEX_W2026\ml-pipelines\lighthouse_csv_v7\lighthouse_csv_v7
Loaded residents: (60, 49) from lighthouse_csv_v7\lighthouse_csv_v7\residents.csv
Loaded safehouses: (9, 13) from lighthouse_csv_v7\lighthouse_csv_v7\safehouses.csv
Loaded health_wellbeing_records: (534, 14) from lighthouse_csv_v7\lighthouse_csv_v7\health_wellbeing_records.csv
Loaded incident_reports: (100, 12) from lighthouse_csv_v7\lighthouse_csv_v7\incident_reports.csv
Loaded process_recordings: (2819, 15) from lighthouse_csv_v7\lighthouse_csv_v7\process_recordings.csv
Loaded intervention_plans: (180, 11) from lighthouse_csv_v7\lighthouse_csv_v7\intervention_plans.csv


In [5]:

# Standardize likely alternative column names across source tables

# Standardize resident aliases.
if not tables["residents"].empty:
    tables["residents"] = rename_if_present(
        tables["residents"],
        {
            "resident_record_id": "resident_id",
            "safehouse_record_id": "safehouse_id",
            "created_date": "intake_date",
        }
    )

# Standardize safehouse aliases.
if not tables["safehouses"].empty:
    tables["safehouses"] = rename_if_present(
        tables["safehouses"],
        {
            "safehouse_record_id": "safehouse_id",
            "safehouse_name": "name",
        }
    )

# Standardize health table aliases.
tables["health_wellbeing_records"] = rename_if_present(
    tables["health_wellbeing_records"],
    {
        "health_record_id": "record_id",
        "resident_record_id": "resident_id",
        "safehouse_record_id": "safehouse_id",
        "record_date": "health_date",
        "assessment_date": "health_date",
        "general_health_score": "raw_health_score",
        "health_score": "raw_health_score",
        "overall_health_score": "raw_health_score",
        "wellbeing_score": "raw_health_score",
        "overall_wellbeing_score": "raw_health_score",
        "sleep_quality_score": "raw_sleep_quality_score",
        "sleep_score": "raw_sleep_quality_score",
        "energy_level_score": "raw_energy_level_score",
        "energy_score": "raw_energy_level_score",
    }
)

# Standardize incident aliases.
if not tables["incident_reports"].empty:
    tables["incident_reports"] = rename_if_present(
        tables["incident_reports"],
        {
            "incident_report_id": "incident_id",
            "resident_record_id": "resident_id",
            "safehouse_record_id": "safehouse_id",
            "record_date": "incident_date",
        }
    )

# Standardize process-recording aliases.
if not tables["process_recordings"].empty:
    tables["process_recordings"] = rename_if_present(
        tables["process_recordings"],
        {
            "process_recording_id": "process_id",
            "resident_record_id": "resident_id",
            "safehouse_record_id": "safehouse_id",
            "record_date": "session_date",
            "session_duration_minutes": "session_duration",
        }
    )

# Standardize intervention-plan aliases.
if not tables["intervention_plans"].empty:
    tables["intervention_plans"] = rename_if_present(
        tables["intervention_plans"],
        {
            "intervention_plan_id": "plan_id",
            "resident_record_id": "resident_id",
            "safehouse_record_id": "safehouse_id",
            "created_date": "plan_created_date",
            "status_date": "plan_status_date",
        }
    )

for table_name, df in tables.items():
    print(f"\n{table_name} columns:")
    print(df.columns.tolist())



residents columns:
['resident_id', 'case_control_no', 'internal_code', 'safehouse_id', 'case_status', 'sex', 'date_of_birth', 'birth_status', 'place_of_birth', 'religion', 'case_category', 'sub_cat_orphaned', 'sub_cat_trafficked', 'sub_cat_child_labor', 'sub_cat_physical_abuse', 'sub_cat_sexual_abuse', 'sub_cat_osaec', 'sub_cat_cicl', 'sub_cat_at_risk', 'sub_cat_street_child', 'sub_cat_child_with_hiv', 'is_pwd', 'pwd_type', 'has_special_needs', 'special_needs_diagnosis', 'family_is_4ps', 'family_solo_parent', 'family_indigenous', 'family_parent_pwd', 'family_informal_settler', 'date_of_admission', 'age_upon_admission', 'present_age', 'length_of_stay', 'referral_source', 'referring_agency_person', 'date_colb_registered', 'date_colb_obtained', 'assigned_social_worker', 'initial_case_assessment', 'date_case_study_prepared', 'reintegration_type', 'reintegration_status', 'initial_risk_level', 'current_risk_level', 'date_enrolled', 'date_closed', 'created_at', 'notes_restricted']

safehouse


### Build the resident-month health base table

This is the core monthly table for the pipeline.  
It standardizes health scores, handles scale differences, and creates one resident-month row per month.


In [6]:

# Build the resident-month health base table

health = tables["health_wellbeing_records"].copy()

if health.empty:
    raise FileNotFoundError("health_wellbeing_records table is empty.")

if "resident_id" not in health.columns:
    raise KeyError("Health table is missing resident_id after alias standardization.")

# Find the date column used for monthly aggregation.
date_col = None
for candidate in ["health_date", "record_date", "assessment_date", "created_at"]:
    if candidate in health.columns:
        date_col = candidate
        break

if date_col is None:
    raise KeyError(
        f"Health table needs a usable date column. Available columns: {health.columns.tolist()}"
    )

health["metric_month"] = month_floor(health[date_col])
health = health.dropna(subset=["resident_id", "metric_month"]).copy()

# Ensure score columns exist before numeric conversion.
for col in ["raw_health_score", "raw_sleep_quality_score", "raw_energy_level_score"]:
    if col not in health.columns:
        health[col] = np.nan

health["raw_health_score"] = safe_numeric(health["raw_health_score"])
health["raw_sleep_quality_score"] = safe_numeric(health["raw_sleep_quality_score"])
health["raw_energy_level_score"] = safe_numeric(health["raw_energy_level_score"])

# If the main health score is missing, create a fallback from the available sub-scores.
component_cols = [c for c in ["raw_sleep_quality_score", "raw_energy_level_score"] if health[c].notna().any()]
if health["raw_health_score"].notna().sum() == 0 and component_cols:
    health["raw_health_score"] = health[component_cols].mean(axis=1)

# Convert 1-5 style health scores to a 0-100 scale if needed.
health_max = health["raw_health_score"].max(skipna=True)
if pd.notna(health_max) and health_max <= 5:
    health["avg_health_score"] = health["raw_health_score"] * 20
else:
    health["avg_health_score"] = health["raw_health_score"]

# Keep sleep and energy on their natural 1-5 style scale.
health["avg_sleep_quality_score"] = health["raw_sleep_quality_score"]
health["avg_energy_level_score"] = health["raw_energy_level_score"]

health_month = (
    health
    .groupby(["resident_id", "metric_month"], as_index=False)
    .agg(
        avg_health_score=("avg_health_score", "mean"),
        avg_sleep_quality_score=("avg_sleep_quality_score", "mean"),
        avg_energy_level_score=("avg_energy_level_score", "mean"),
    )
)

health_counts = (
    health
    .groupby(["resident_id", "metric_month"], as_index=False)
    .size()
    .rename(columns={"size": "health_record_count"})
)

health_month = health_month.merge(
    health_counts,
    on=["resident_id", "metric_month"],
    how="left"
)

# Add safehouse_id when it is available in the health table.
if "safehouse_id" in health.columns:
    safehouse_lookup = (
        health[["resident_id", "metric_month", "safehouse_id"]]
        .dropna(subset=["resident_id", "metric_month"])
        .drop_duplicates(subset=["resident_id", "metric_month"])
    )
    health_month = health_month.merge(
        safehouse_lookup,
        on=["resident_id", "metric_month"],
        how="left"
    )

print("health_month columns:", health_month.columns.tolist())
display(health_month.head())


health_month columns: ['resident_id', 'metric_month', 'avg_health_score', 'avg_sleep_quality_score', 'avg_energy_level_score', 'health_record_count']


,resident_id,metric_month,avg_health_score,avg_sleep_quality_score,avg_energy_level_score,health_record_count
0,1,2023-10-01,61.8,3.18,2.90,1
1,1,2023-11-01,61.0,3.18,2.85,1
2,1,2023-12-01,61.0,3.19,2.94,1
3,1,2024-01-01,61.6,3.21,2.92,1
4,1,2024-02-01,62.6,3.26,2.93,1



### Supporting monthly feature tables

The next few blocks summarize supporting case-management activity by resident-month:
- incidents
- process recordings
- intervention plans

These do not define the target directly, but they help describe the context around wellbeing trajectories.


In [7]:

# Build monthly incident features

incidents = tables["incident_reports"].copy()

if incidents.empty:
    incident_month = health_month[["resident_id", "metric_month"]].copy()
    incident_month["incident_count"] = 0
    incident_month["high_severity_incidents"] = 0
    incident_month["medium_severity_incidents"] = 0
    incident_month["follow_up_incidents"] = 0
else:
    if "incident_date" not in incidents.columns:
        raise KeyError("Incident table needs incident_date after alias standardization.")

    incidents["metric_month"] = month_floor(incidents["incident_date"])
    incidents = incidents.dropna(subset=["resident_id", "metric_month"]).copy()

    if "severity" in incidents.columns:
        incidents["severity"] = incidents["severity"].astype(str).str.strip().str.lower()
    else:
        incidents["severity"] = ""

    if "follow_up_required" in incidents.columns:
        incidents["follow_up_flag"] = (
            incidents["follow_up_required"]
            .astype(str)
            .str.strip()
            .str.lower()
            .isin(["true", "1", "yes", "y"])
            .astype(int)
        )
    else:
        incidents["follow_up_flag"] = 0

    incident_month = (
        incidents
        .groupby(["resident_id", "metric_month"], as_index=False)
        .agg(
            incident_count=("resident_id", "size"),
            high_severity_incidents=("severity", lambda s: (s == "high").sum()),
            medium_severity_incidents=("severity", lambda s: (s == "medium").sum()),
            follow_up_incidents=("follow_up_flag", "sum"),
        )
    )

display(incident_month.head())


,resident_id,metric_month,incident_count,high_severity_incidents,medium_severity_incidents,follow_up_incidents
0,1,2024-02-01,1,0,0,0
1,1,2024-06-01,1,0,1,0
2,1,2025-04-01,1,0,0,0
3,1,2026-02-01,1,1,0,1
4,3,2025-02-01,1,0,0,1


In [8]:

# Build monthly process recording / counseling features

process = tables["process_recordings"].copy()

if process.empty:
    process_month = health_month[["resident_id", "metric_month"]].copy()
    process_month["process_session_count"] = 0
    process_month["avg_session_duration"] = 0.0
    process_month["group_session_share"] = 0.0
else:
    if "session_date" not in process.columns:
        raise KeyError("Process recordings table needs session_date after alias standardization.")

    process["metric_month"] = month_floor(process["session_date"])
    process = process.dropna(subset=["resident_id", "metric_month"]).copy()

    if "session_duration" not in process.columns:
        process["session_duration"] = np.nan
    process["session_duration"] = safe_numeric(process["session_duration"])

    if "session_type" in process.columns:
        process["group_session_flag"] = (
            process["session_type"].astype(str).str.strip().str.lower().eq("group").astype(int)
        )
    else:
        process["group_session_flag"] = 0

    process_month = (
        process
        .groupby(["resident_id", "metric_month"], as_index=False)
        .agg(
            process_session_count=("resident_id", "size"),
            avg_session_duration=("session_duration", "mean"),
            group_session_share=("group_session_flag", "mean"),
        )
    )

display(process_month.head())


,resident_id,metric_month,process_session_count,avg_session_duration,group_session_share
0,1,2023-11-01,3,74.000000,0.333333
1,1,2023-12-01,6,75.333333,0.833333
2,1,2024-01-01,3,60.666667,0.333333
3,1,2024-02-01,3,44.333333,0.333333
4,1,2024-03-01,5,71.400000,0.600000


In [9]:

# Build monthly intervention plan features

plans = tables["intervention_plans"].copy()

if plans.empty:
    plan_month = health_month[["resident_id", "metric_month"]].copy()
    plan_month["plans_created"] = 0
    plan_month["achieved_plans"] = 0
else:
    if "plan_created_date" not in plans.columns:
        for candidate in ["created_at", "created_date", "plan_date", "date_created", "intervention_date"]:
            if candidate in plans.columns:
                plans = plans.rename(columns={candidate: "plan_created_date"})
                break

    if "plan_created_date" not in plans.columns:
        raise KeyError(
            f"Intervention plans table needs a usable date column. Available columns: {plans.columns.tolist()}"
        )

    plans["metric_month"] = month_floor(plans["plan_created_date"])
    plans = plans.dropna(subset=["resident_id", "metric_month"]).copy()

    if "completion_status" in plans.columns:
        status_series = plans["completion_status"]
    elif "status" in plans.columns:
        status_series = plans["status"]
    else:
        status_series = pd.Series("", index=plans.index)

    plans["achieved_flag"] = (
        status_series.astype(str)
        .str.strip()
        .str.lower()
        .isin(["achieved", "completed", "complete", "met"])
        .astype(int)
    )

    plan_month = (
        plans
        .groupby(["resident_id", "metric_month"], as_index=False)
        .agg(
            plans_created=("resident_id", "size"),
            achieved_plans=("achieved_flag", "sum"),
        )
    )

display(plan_month.head())


,resident_id,metric_month,plans_created,achieved_plans
0,1,2023-10-01,3,0
1,2,2023-03-01,3,1
2,3,2024-05-01,3,0
3,4,2024-09-01,3,0
4,5,2024-01-01,3,1


In [10]:

# Merge all monthly feature tables into one resident-month panel

resident_month_panel = health_month.copy()

for frame in [incident_month, process_month, plan_month]:
    resident_month_panel = resident_month_panel.merge(
        frame,
        on=["resident_id", "metric_month"],
        how="left"
    )

# Add resident-level context if available.
residents = tables["residents"].copy()
if not residents.empty and "resident_id" in residents.columns:
    resident_keep_cols = [col for col in ["resident_id", "safehouse_id", "name"] if col in residents.columns]
    if resident_keep_cols:
        residents_small = residents[resident_keep_cols].drop_duplicates(subset=["resident_id"])
        resident_month_panel = resident_month_panel.merge(
            residents_small,
            on="resident_id",
            how="left",
            suffixes=("", "_resident")
        )

# Add safehouse metadata if available.
safehouses = tables["safehouses"].copy()
if not safehouses.empty and "safehouse_id" in safehouses.columns:
    safehouse_keep_cols = [col for col in ["safehouse_id", "region", "open_date", "name"] if col in safehouses.columns]
    if safehouse_keep_cols:
        safehouses_small = safehouses[safehouse_keep_cols].drop_duplicates(subset=["safehouse_id"])
        resident_month_panel = resident_month_panel.merge(
            safehouses_small,
            on="safehouse_id",
            how="left",
            suffixes=("", "_safehouse")
        )

# Fill count-like fields after the monthly merges.
for col in [
    "health_record_count",
    "incident_count",
    "high_severity_incidents",
    "medium_severity_incidents",
    "follow_up_incidents",
    "process_session_count",
    "plans_created",
    "achieved_plans",
]:
    if col in resident_month_panel.columns:
        resident_month_panel[col] = resident_month_panel[col].fillna(0)

for col in ["avg_session_duration", "group_session_share"]:
    if col in resident_month_panel.columns:
        resident_month_panel[col] = safe_numeric(resident_month_panel[col]).fillna(0)

if "open_date" in resident_month_panel.columns:
    resident_month_panel["open_date"] = pd.to_datetime(resident_month_panel["open_date"], errors="coerce")
    resident_month_panel["months_since_open"] = (
        (pd.to_datetime(resident_month_panel["metric_month"]) - resident_month_panel["open_date"])
        / np.timedelta64(30, "D")
    )

display(resident_month_panel.head())
print("resident_month_panel columns:", resident_month_panel.columns.tolist())


,resident_id,metric_month,avg_health_score,avg_sleep_quality_score,avg_energy_level_score,health_record_count,incident_count,high_severity_incidents,medium_severity_incidents,follow_up_incidents,process_session_count,avg_session_duration,group_session_share,plans_created,achieved_plans,safehouse_id,region,open_date,name,months_since_open
0,1,2023-10-01,61.8,3.18,2.90,1,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,3.0,0.0,4,Visayas,2022-05-16,Lighthouse Safehouse 4,16.766667
1,1,2023-11-01,61.0,3.18,2.85,1,0.0,0.0,0.0,0.0,3.0,74.000000,0.333333,0.0,0.0,4,Visayas,2022-05-16,Lighthouse Safehouse 4,17.800000
2,1,2023-12-01,61.0,3.19,2.94,1,0.0,0.0,0.0,0.0,6.0,75.333333,0.833333,0.0,0.0,4,Visayas,2022-05-16,Lighthouse Safehouse 4,18.800000
3,1,2024-01-01,61.6,3.21,2.92,1,0.0,0.0,0.0,0.0,3.0,60.666667,0.333333,0.0,0.0,4,Visayas,2022-05-16,Lighthouse Safehouse 4,19.833333
4,1,2024-02-01,62.6,3.26,2.93,1,1.0,0.0,0.0,0.0,3.0,44.333333,0.333333,0.0,0.0,4,Visayas,2022-05-16,Lighthouse Safehouse 4,20.866667


resident_month_panel columns: ['resident_id', 'metric_month', 'avg_health_score', 'avg_sleep_quality_score', 'avg_energy_level_score', 'health_record_count', 'incident_count', 'high_severity_incidents', 'medium_severity_incidents', 'follow_up_incidents', 'process_session_count', 'avg_session_duration', 'group_session_share', 'plans_created', 'achieved_plans', 'safehouse_id', 'region', 'open_date', 'name', 'months_since_open']



## 3. Modeling & Feature Selection

### Create the future target carefully
The target must come from the **next month**, while the predictors come from the **current month**.

That means:
- shifting future outcomes forward within each resident history
- keeping only rows with a true next-month record
- dropping rows that cannot support a real next-month prediction task


In [11]:

# Create next-month target and future health outcomes

panel = resident_month_panel.copy()
panel = panel.sort_values(["resident_id", "metric_month"]).reset_index(drop=True)

required_target_cols = [
    "avg_health_score",
    "avg_sleep_quality_score",
    "avg_energy_level_score"
]
missing_target_cols = [col for col in required_target_cols if col not in panel.columns]
if missing_target_cols:
    raise KeyError(f"Missing required columns for target creation: {missing_target_cols}")

panel["next_month_health_score"] = panel.groupby("resident_id")["avg_health_score"].shift(-1)
panel["next_month_sleep_quality_score"] = panel.groupby("resident_id")["avg_sleep_quality_score"].shift(-1)
panel["next_month_energy_level_score"] = panel.groupby("resident_id")["avg_energy_level_score"].shift(-1)
panel["next_metric_month"] = panel.groupby("resident_id")["metric_month"].shift(-1)

# Only keep labels where the next record is exactly the next calendar month.
expected_next_month = panel["metric_month"] + pd.offsets.MonthBegin(1)
panel["has_true_next_month"] = panel["next_metric_month"].eq(expected_next_month)

# Build the target using only outcome dimensions that are actually populated.
available_outcome_conditions = []

if panel["next_month_health_score"].notna().any():
    available_outcome_conditions.append(panel["next_month_health_score"].ge(70))

if panel["next_month_sleep_quality_score"].notna().any():
    available_outcome_conditions.append(panel["next_month_sleep_quality_score"].ge(3))

if panel["next_month_energy_level_score"].notna().any():
    available_outcome_conditions.append(panel["next_month_energy_level_score"].ge(3))

if not available_outcome_conditions:
    raise ValueError("No usable next-month outcome columns were available to build the target.")

combined_condition = available_outcome_conditions[0]
for condition in available_outcome_conditions[1:]:
    combined_condition = combined_condition & condition

panel["positive_health_trajectory_next_month"] = np.where(
    panel["has_true_next_month"] & combined_condition,
    1,
    np.where(panel["has_true_next_month"], 0, np.nan)
)

display(
    panel[
        [
            "resident_id",
            "metric_month",
            "avg_health_score",
            "avg_sleep_quality_score",
            "avg_energy_level_score",
            "next_month_health_score",
            "next_month_sleep_quality_score",
            "next_month_energy_level_score",
            "positive_health_trajectory_next_month",
        ]
    ].head(12)
)


,resident_id,metric_month,avg_health_score,avg_sleep_quality_score,avg_energy_level_score,next_month_health_score,next_month_sleep_quality_score,next_month_energy_level_score,positive_health_trajectory_next_month
0,1,2023-10-01,61.8,3.18,2.90,61.0,3.18,2.85,0.0
1,1,2023-11-01,61.0,3.18,2.85,61.0,3.19,2.94,0.0
2,1,2023-12-01,61.0,3.19,2.94,61.6,3.21,2.92,0.0
3,1,2024-01-01,61.6,3.21,2.92,62.6,3.26,2.93,0.0
4,1,2024-02-01,62.6,3.26,2.93,64.4,3.20,2.91,0.0
5,1,2024-03-01,64.4,3.20,2.91,NaN,NaN,NaN,NaN
6,2,2023-03-01,63.8,3.24,2.84,66.8,3.25,2.86,0.0
7,2,2023-04-01,66.8,3.25,2.86,67.4,3.33,2.86,0.0
8,2,2023-05-01,67.4,3.33,2.86,67.0,3.39,2.96,0.0
9,2,2023-06-01,67.0,3.39,2.96,69.6,3.40,2.99,0.0


In [12]:

# Add lag and rolling context features for the model

panel = panel.sort_values(["resident_id", "metric_month"]).reset_index(drop=True)

lag_features = [
    "avg_health_score",
    "avg_sleep_quality_score",
    "avg_energy_level_score",
    "incident_count",
    "high_severity_incidents",
    "process_session_count",
    "plans_created",
    "achieved_plans",
]

for col in lag_features:
    if col in panel.columns:
        panel[f"{col}_lag1"] = panel.groupby("resident_id")[col].shift(1)
        panel[f"{col}_lag3_mean"] = (
            panel.groupby("resident_id")[col]
            .transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
        )

panel["metric_month"] = pd.to_datetime(panel["metric_month"], errors="coerce")
panel["month_num"] = panel["metric_month"].dt.month
panel["year_num"] = panel["metric_month"].dt.year

# Keep only rows with a known target.
model_df = panel.dropna(subset=["positive_health_trajectory_next_month"]).copy()
model_df["positive_health_trajectory_next_month"] = model_df["positive_health_trajectory_next_month"].astype(int)

print("Modeling shape:", model_df.shape)
print(model_df["positive_health_trajectory_next_month"].value_counts(dropna=False))
display(model_df.head())


Modeling shape: (474, 44)
positive_health_trajectory_next_month
0    441
1     33
Name: count, dtype: int64


,resident_id,metric_month,avg_health_score,avg_sleep_quality_score,avg_energy_level_score,health_record_count,incident_count,high_severity_incidents,medium_severity_incidents,follow_up_incidents,...,high_severity_incidents_lag1,high_severity_incidents_lag3_mean,process_session_count_lag1,process_session_count_lag3_mean,plans_created_lag1,plans_created_lag3_mean,achieved_plans_lag1,achieved_plans_lag3_mean,month_num,year_num
0,1,2023-10-01,61.8,3.18,2.90,1,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10,2023
1,1,2023-11-01,61.0,3.18,2.85,1,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,3.0,3.0,0.0,0.0,11,2023
2,1,2023-12-01,61.0,3.19,2.94,1,0.0,0.0,0.0,0.0,...,0.0,0.0,3.0,1.5,0.0,1.5,0.0,0.0,12,2023
3,1,2024-01-01,61.6,3.21,2.92,1,0.0,0.0,0.0,0.0,...,0.0,0.0,6.0,3.0,0.0,1.0,0.0,0.0,1,2024
4,1,2024-02-01,62.6,3.26,2.93,1,1.0,0.0,0.0,0.0,...,0.0,0.0,3.0,4.0,0.0,0.0,0.0,0.0,2,2024


In [13]:

# Define features, label, and a safer grouped train/test split

target_col = "positive_health_trajectory_next_month"

drop_cols = [
    target_col,
    "next_month_health_score",
    "next_month_sleep_quality_score",
    "next_month_energy_level_score",
    "next_metric_month",
    "has_true_next_month",
    "metric_month",
    "open_date",
]

X = model_df.drop(columns=[col for col in drop_cols if col in model_df.columns]).copy()
y = model_df[target_col].copy()
groups = model_df["resident_id"].copy()

if y.nunique() < 2:
    raise ValueError(
        "The modeled target contains only one class after target creation. "
        "This usually means the target threshold is too strict or the data does not contain enough next-month variation."
    )

# Try multiple grouped splits until both train and test contain both classes.
split_found = False
for seed in [27, 42, 73, 101, 202]:
    gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=seed)
    train_idx, test_idx = next(gss.split(X, y, groups=groups))

    y_train_candidate = y.iloc[train_idx]
    y_test_candidate = y.iloc[test_idx]

    if y_train_candidate.nunique() == 2 and y_test_candidate.nunique() == 2:
        split_found = True
        break

if not split_found:
    raise ValueError(
        "Could not find a grouped train/test split with both classes present in train and test. "
        "The dataset may be too small or too imbalanced for the current target."
    )

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()
y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()
groups_train = groups.iloc[train_idx].copy()

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("y_train distribution:")
print(y_train.value_counts(dropna=False))
print("y_test distribution:")
print(y_test.value_counts(dropna=False))

# Remove raw datetime columns before preprocessing.
datetime_cols_train = X_train.select_dtypes(include=["datetime64[ns]", "datetime64", "datetimetz"]).columns.tolist()
if datetime_cols_train:
    X_train = X_train.drop(columns=datetime_cols_train, errors="ignore")
    X_test = X_test.drop(columns=datetime_cols_train, errors="ignore")

numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)


Train shape: (371, 36)
Test shape: (103, 36)
y_train distribution:
positive_health_trajectory_next_month
0    351
1     20
Name: count, dtype: int64
y_test distribution:
positive_health_trajectory_next_month
0    90
1    13
Name: count, dtype: int64
Numeric features: ['resident_id', 'avg_health_score', 'avg_sleep_quality_score', 'avg_energy_level_score', 'health_record_count', 'incident_count', 'high_severity_incidents', 'medium_severity_incidents', 'follow_up_incidents', 'process_session_count', 'avg_session_duration', 'group_session_share', 'plans_created', 'achieved_plans', 'safehouse_id', 'months_since_open', 'avg_health_score_lag1', 'avg_health_score_lag3_mean', 'avg_sleep_quality_score_lag1', 'avg_sleep_quality_score_lag3_mean', 'avg_energy_level_score_lag1', 'avg_energy_level_score_lag3_mean', 'incident_count_lag1', 'incident_count_lag3_mean', 'high_severity_incidents_lag1', 'high_severity_incidents_lag3_mean', 'process_session_count_lag1', 'process_session_count_lag3_mean', 'pl

In [14]:

# Build preprocessing and candidate model pipelines

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

logit_pipeline = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
])

rf_pipeline = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=4,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])



## 4. Evaluation & Interpretation

This section cross-validates the candidate models on the training data and then evaluates the selected model on the held-out test set.

### Why the earlier notebook failed
The earlier version crashed because one cross-validation fold contained only a single class.  
This revised notebook guards against that by:
- searching for a grouped split where both train and test contain both classes
- shrinking the number of CV folds if the minority class is small
- skipping invalid fold counts rather than crashing unexpectedly


In [15]:

# Cross-validate both candidate models on the training data

scoring = {
    "roc_auc": "roc_auc",
    "avg_precision": "average_precision",
    "f1": "f1",
    "precision": "precision",
    "recall": "recall",
}

# Limit CV folds to the available minority-class count.
min_class_count = int(y_train.value_counts().min())
cv_splits = min(5, min_class_count)

if cv_splits < 2:
    raise ValueError(
        "Training data does not contain enough minority-class examples for valid cross-validation. "
        "Consider relaxing the target threshold or using more data."
    )

cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=42)

logit_cv = cross_validate(
    logit_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=None
)

rf_cv = cross_validate(
    rf_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=None
)

cv_results = pd.DataFrame({
    "model": ["LogisticRegression", "RandomForest"],
    "cv_roc_auc_mean": [logit_cv["test_roc_auc"].mean(), rf_cv["test_roc_auc"].mean()],
    "cv_avg_precision_mean": [logit_cv["test_avg_precision"].mean(), rf_cv["test_avg_precision"].mean()],
    "cv_f1_mean": [logit_cv["test_f1"].mean(), rf_cv["test_f1"].mean()],
    "cv_precision_mean": [logit_cv["test_precision"].mean(), rf_cv["test_precision"].mean()],
    "cv_recall_mean": [logit_cv["test_recall"].mean(), rf_cv["test_recall"].mean()],
})

cv_results = cv_results.sort_values("cv_roc_auc_mean", ascending=False).reset_index(drop=True)
display(cv_results)


,model,cv_roc_auc_mean,cv_avg_precision_mean,cv_f1_mean,cv_precision_mean,cv_recall_mean
0,RandomForest,0.955926,0.609888,0.554762,0.563333,0.60
1,LogisticRegression,0.885362,0.435018,0.351399,0.260714,0.55


In [16]:

# Fit the better candidate and evaluate on the held-out test set

best_model_name = cv_results.loc[0, "model"]
best_pipeline = rf_pipeline if best_model_name == "RandomForest" else logit_pipeline

best_pipeline.fit(X_train, y_train)

test_pred = best_pipeline.predict(X_test)
test_proba = best_pipeline.predict_proba(X_test)[:, 1]

metrics_summary = pd.DataFrame([{
    "selected_model": best_model_name,
    "test_accuracy": accuracy_score(y_test, test_pred),
    "test_roc_auc": roc_auc_score(y_test, test_proba),
    "test_avg_precision": average_precision_score(y_test, test_proba),
    "test_f1": f1_score(y_test, test_pred),
    "test_precision": precision_score(y_test, test_pred, zero_division=0),
    "test_recall": recall_score(y_test, test_pred, zero_division=0),
}])

display(metrics_summary)
print("Confusion matrix:")
print(confusion_matrix(y_test, test_pred))
print("\nClassification report:")
print(classification_report(y_test, test_pred, zero_division=0))


,selected_model,test_accuracy,test_roc_auc,test_avg_precision,test_f1,test_precision,test_recall
0,RandomForest,0.941748,0.976068,0.873446,0.727273,0.888889,0.615385


Confusion matrix:
[[89  1]
 [ 5  8]]

Classification report:
              precision    recall  f1-score   support

           0       0.95      0.99      0.97        90
           1       0.89      0.62      0.73        13

    accuracy                           0.94       103
   macro avg       0.92      0.80      0.85       103
weighted avg       0.94      0.94      0.94       103




## 5. Causal and Relationship Analysis

This section is for interpretation, not causal proof.

The model can help show:
- whether prior health scores matter strongly
- whether incidents or counseling intensity appear associated with next-month outcomes
- whether intervention planning correlates with better trajectories

Those are **associations useful for monitoring**, not defensible causal claims by themselves.


In [17]:

# Measure permutation importance on the held-out test set

perm = permutation_importance(
    best_pipeline,
    X_test,
    y_test,
    n_repeats=15,
    random_state=42,
    scoring="roc_auc"
)

importance_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)

display(importance_df.head(15))


,feature,importance_mean,importance_std
1,avg_health_score,0.070541,0.022701
19,avg_health_score_lag3_mean,0.007123,0.003400
18,avg_health_score_lag1,0.005926,0.006726
15,region,0.001880,0.001368
20,avg_sleep_quality_score_lag1,0.001766,0.003606
21,avg_sleep_quality_score_lag3_mean,0.001709,0.002162
25,incident_count_lag3_mean,0.001481,0.000581
28,process_session_count_lag1,0.001368,0.001279
29,process_session_count_lag3_mean,0.001254,0.001459
16,name,0.001140,0.000864



## 6. Deployment Notes

The application does not need to retrain this model live during page load.

Instead, the pipeline should:
1. generate a scored output table or CSV
2. let the backend read that artifact
3. surface the results in the website

### Best page fit
- **Caseload Inventory**: resident-level risk/probability field
- **Admin Dashboard**: summary counts and high-priority queue
- **Reports & Analytics**: aggregate trend charts and outcome summaries


In [18]:

# Score the full labeled modeling frame for downstream dashboard use

X_full = model_df.drop(columns=[col for col in drop_cols if col in model_df.columns]).copy()
datetime_cols_full = X_full.select_dtypes(include=["datetime64[ns]", "datetime64", "datetimetz"]).columns.tolist()
if datetime_cols_full:
    X_full = X_full.drop(columns=datetime_cols_full, errors="ignore")

full_proba = best_pipeline.predict_proba(X_full)[:, 1]
full_pred = best_pipeline.predict(X_full)

scores = model_df[["resident_id", "metric_month", "safehouse_id"]].copy()
scores["positive_health_trajectory_next_month_actual"] = model_df[target_col].values
scores["positive_health_trajectory_next_month_pred"] = full_pred
scores["positive_health_trajectory_next_month_score"] = full_proba

output_path = OUTPUT_DIR / "health_wellbeing_trajectory_scores.csv"
scores.to_csv(output_path, index=False)

print(f"Saved live scoring output to: {output_path}")
display(scores.head(15))


Saved live scoring output to: generated_outputs\health_wellbeing_trajectory_scores.csv


,resident_id,metric_month,safehouse_id,positive_health_trajectory_next_month_actual,positive_health_trajectory_next_month_pred,positive_health_trajectory_next_month_score
0,1,2023-10-01,4,0,0,0.015379
1,1,2023-11-01,4,0,0,0.015972
2,1,2023-12-01,4,0,0,0.013258
3,1,2024-01-01,4,0,0,0.011090
4,1,2024-02-01,4,0,0,0.021277
6,2,2023-03-01,3,0,0,0.009465
7,2,2023-04-01,3,0,0,0.030904
8,2,2023-05-01,3,0,0,0.099202
9,2,2023-06-01,3,0,0,0.140693
10,2,2023-07-01,3,0,0,0.397108


In [19]:

# Save the trained pipeline artifact and a small metadata summary

artifact = {
    "model_name": best_model_name,
    "pipeline": best_pipeline,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "target_definition": {
        "next_month_avg_health_score_min": 70,
        "next_month_avg_sleep_quality_score_min": 3,
        "next_month_avg_energy_level_score_min": 3,
    },
}

artifact_path = OUTPUT_DIR / "health_wellbeing_trajectory_model.joblib"
joblib.dump(artifact, artifact_path)

metadata_path = OUTPUT_DIR / "health_wellbeing_trajectory_metrics.csv"
metrics_summary.to_csv(metadata_path, index=False)

print(f"Saved model artifact to: {artifact_path}")
print(f"Saved metrics summary to: {metadata_path}")


Saved model artifact to: generated_outputs\health_wellbeing_trajectory_model.joblib
Saved metrics summary to: generated_outputs\health_wellbeing_trajectory_metrics.csv
